In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-flat-kp2000kd50'  # ckpt = 20000
# exp_name = 'friction-walking-fractal-kp2000kd50-action_rate-0.01'  # ckpt = 20000
# exp_name = 'friction-walking-fractal-kp2000kd50-linvel-4'
# exp_name = 'friction-walking-fractal-kp2000kd50-all0.1'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.9

In [9]:
env_cfg

{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 0.25,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.9,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.1, 2.0],
  'restitution': [0.0, 0.5],
  'kp': [15000.0, 25000.0],
  'kd': [40.0, 80.0

In [10]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [11]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [12]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.0544, -0.0739,  0.6372, -0.1969, -0.6360, -0.0727, -0.0386,  0.2567,
          0.3708, -0.1026, -0.4911, -0.2027]], device='cuda:0')
Scaled actions :  tensor([[ 0.0544, -0.0739,  0.6372, -0.1969, -0.6360, -0.0727, -0.0386,  0.2567,
          0.3708, -0.1026, -0.4911, -0.2027]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[ 1.2169e-05, -1.2488e-02,  1.7362e-05,  1.1224e-10, -1.6308e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -1.1109e-07,
         -1.5817e-07, -4.7266e-05,  2.2531e-04, -1.2803e-04,  2.3233e-07,
          1.6094e-07, -7.5484e-09, -4.7386e-05,  2.2554e-04, -1.2821e-04,
         -1.3234e-06, -5.5543e-06, -7.9086e-06, -2.3640e-03,  1.1269e-02,
         -6.4021e-03,  1.1617e-05,  8.0469e-06, -3.7742e-07, -2.3695e-03,
          1.1281e-02, -6.4100e-03, -6.6171e-05,  5.4394e-02, -7.3856e-02,
          6.3719e-01, -1.9691e-01, -6.3601e-01, -7.2731e-02, -3.8605e-02,
          2.5674e-01,  3.7082e-01, -1.0260e-01, -4.9107e-01, -2.0265e-01]],
       device='cuda:0')
torques: [-2.97352313e-17  3.47214850e-16 -1.31003155e-05  2.93246369e-05
 -1.25008273e-05  2.51737627e-16 -2.85411999e-17 -9.57523010e-17
 -1.31003155e-05  2.93246369e-05 -1.25008273e-05 -4.99118832e-17]
データ収集: step 2


In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.0554, -0.3076,  0.5089, -0.1670, -0.9032, -0.1646, -0.2142,  0.2690,
          0.3135, -0.3396, -0.6043, -0.3125]], device='cuda:0')
Scaled actions :  tensor([[ 0.0554, -0.3076,  0.5089, -0.1670, -0.9032, -0.1646, -0.2142,  0.2690,
          0.3135, -0.3396, -0.6043, -0.3125]], device='cuda:0')
obs :  tensor([[-3.3873e-02, -2.2546e-01, -3.5590e-02, -5.3000e-03,  7.7697e-04,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  8.0978e-04,
          1.3808e-03,  8.2843e-03,  2.1533e-03, -1.2880e-02, -5.9786e-04,
         -4.7201e-04,  2.1038e-03,  8.1316e-03,  2.2240e-03, -1.2602e-02,
         -5.0642e-03,  7.5475e-03,  1.2173e-02,  7.6996e-02,  8.1800e-03,
         -1.1076e-01, -5.3443e-03, -4.1433e-03,  1.8866e-02,  7.5736e-02,
          8.7986e-03, -1.0669e-01, -4.4992e-02,  5.5386e-02, -3.0757e-01,
          5.0894e-01, -1.6701e-01, -9.0322e-01, -1.6460e-01, -2.1421e-01,
          2.6899e-01,  3.1349e-01, -3.3957e-01, -6.0430e-01, -3.1

In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.0178, -0.3889,  0.2717, -0.3602, -0.9145, -0.2828, -0.0256,  0.3172,
         -0.1767, -0.4554, -0.8634, -0.2999]], device='cuda:0')
Scaled actions :  tensor([[ 0.0178, -0.3889,  0.2717, -0.3602, -0.9145, -0.2828, -0.0256,  0.3172,
         -0.1767, -0.4554, -0.8634, -0.2999]], device='cuda:0')
obs :  tensor([[-0.0309, -0.3394, -0.0164, -0.0173,  0.0021, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0024,  0.0032,  0.0303,  0.0028, -0.0463, -0.0041, -0.0055,
          0.0056,  0.0297,  0.0033, -0.0435, -0.0207,  0.0081,  0.0051,  0.1306,
         -0.0011, -0.2027, -0.0257, -0.0397,  0.0168,  0.1267,  0.0021, -0.1810,
         -0.0968,  0.0178, -0.3889,  0.2717, -0.3602, -0.9145, -0.2828, -0.0256,
          0.3172, -0.1767, -0.4554, -0.8634, -0.2999]], device='cuda:0')
torques: [   6.923853   -174.36854168  -60.39476692  -90.81290329   32.35426829
  -26.37954269  -22.36999864   88.15610116 -152.73891793 -182.67599983
  148.66292645   78.02777036

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-5.3764e-02, -3.4165e-01,  3.0317e-01, -4.9042e-01, -8.6820e-01,
         -2.5748e-01,  6.4261e-04,  3.5319e-01, -8.2231e-02, -5.3046e-01,
         -7.4764e-01, -2.5039e-01]], device='cuda:0')
Scaled actions :  tensor([[-5.3764e-02, -3.4165e-01,  3.0317e-01, -4.9042e-01, -8.6820e-01,
         -2.5748e-01,  6.4261e-04,  3.5319e-01, -8.2231e-02, -5.3046e-01,
         -7.4764e-01, -2.5039e-01]], device='cuda:0')
obs :  tensor([[ 2.8413e-02, -1.9066e-01, -1.1915e-01, -2.7839e-02,  1.9184e-03,
         -9.9961e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.5394e-03,
          3.3817e-03,  5.4472e-02,  7.3302e-04, -9.2342e-02, -1.2614e-02,
         -1.0681e-02,  8.3647e-03,  5.0884e-02,  2.8762e-03, -8.4766e-02,
         -4.0421e-02, -4.2273e-03, -3.8699e-03,  1.0735e-01, -1.7355e-02,
         -2.3981e-01, -5.3159e-02, -1.4934e-02,  1.2081e-02,  8.6229e-02,
         -7.8405e-03, -2.1553e-01, -9.3258e-02, -5.3764e-02, -3.4165e-01,
          3.0317e-01, -4.

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.1572, -0.0506,  0.6908, -0.5254, -0.7027, -0.1230, -0.0794,  0.4558,
          0.2922, -0.2940, -0.3520, -0.2816]], device='cuda:0')
Scaled actions :  tensor([[-0.1572, -0.0506,  0.6908, -0.5254, -0.7027, -0.1230, -0.0794,  0.4558,
          0.2922, -0.2940, -0.3520, -0.2816]], device='cuda:0')
obs :  tensor([[ 5.5359e-02, -2.3761e-02, -1.8639e-01, -3.1802e-02, -5.1770e-05,
         -9.9949e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.9950e-04,
          1.3852e-03,  7.2595e-02, -5.0054e-03, -1.3961e-01, -2.3407e-02,
         -1.0843e-02,  1.1474e-02,  6.3091e-02, -9.3389e-04, -1.2580e-01,
         -5.5887e-02, -2.1416e-02, -1.5020e-02,  7.6010e-02, -3.8289e-02,
         -2.2730e-01, -5.1907e-02,  9.4548e-03,  1.8849e-02,  4.0296e-02,
         -2.8716e-02, -1.9495e-01, -6.1933e-02, -1.5716e-01, -5.0627e-02,
          6.9081e-01, -5.2540e-01, -7.0274e-01, -1.2302e-01, -7.9399e-02,
          4.5581e-01,  2.9220e-01, -2.9397e-01, -3.5204e-01, -2.8

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-0.1401, -0.0812,  0.9265, -0.4124, -0.2830, -0.0613, -0.0379,  0.2591,
          0.3584,  0.0207, -0.1408, -0.3246]], device='cuda:0')
Scaled actions :  tensor([[-0.1401, -0.0812,  0.9265, -0.4124, -0.2830, -0.0613, -0.0379,  0.2591,
          0.3584,  0.0207, -0.1408, -0.3246]], device='cuda:0')
obs :  tensor([[-7.2780e-03, -8.6325e-02, -2.4507e-01, -3.4484e-02, -1.1713e-03,
         -9.9940e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.2420e-03,
         -5.3670e-04,  9.2433e-02, -1.5069e-02, -1.8462e-01, -3.1457e-02,
         -9.6653e-03,  2.0294e-02,  7.4471e-02, -1.0491e-02, -1.5740e-01,
         -6.8077e-02, -5.0523e-02, -4.9355e-03,  1.1120e-01, -5.8886e-02,
         -2.1695e-01, -3.0372e-02,  2.7597e-03,  6.3849e-02,  6.5854e-02,
         -6.2420e-02, -1.2344e-01, -5.8231e-02, -1.4009e-01, -8.1204e-02,
          9.2653e-01, -4.1236e-01, -2.8299e-01, -6.1316e-02, -3.7898e-02,
          2.5915e-01,  3.5842e-01,  2.0707e-02, -1.4083e-01, -3.2

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-0.1744,  0.0513,  0.9175, -0.5244,  0.0599, -0.0329,  0.0616,  0.1269,
         -0.1265,  0.1797, -0.2658, -0.2465]], device='cuda:0')
Scaled actions :  tensor([[-0.1744,  0.0513,  0.9175, -0.5244,  0.0599, -0.0329,  0.0616,  0.1269,
         -0.1265,  0.1797, -0.2658, -0.2465]], device='cuda:0')
obs :  tensor([[ 0.0198, -0.1840, -0.3368, -0.0402, -0.0018, -0.9992,  1.0000,  0.0000,
          0.0000, -0.0193, -0.0031,  0.1185, -0.0288, -0.2160, -0.0338, -0.0090,
          0.0335,  0.0882, -0.0197, -0.1699, -0.0787, -0.0561, -0.0207,  0.1452,
         -0.0766, -0.1080,  0.0031,  0.0035,  0.0637,  0.0662, -0.0287, -0.0124,
         -0.0437, -0.1744,  0.0513,  0.9175, -0.5244,  0.0599, -0.0329,  0.0616,
          0.1269, -0.1265,  0.1797, -0.2658, -0.2465]], device='cuda:0')
torques: [ -35.84883852  -35.92359727  200.         -154.57571988  200.
   37.05244856   -0.59245321    0.5591404   -60.62416717   81.10033349
  200.           39.17757552]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-0.2560,  0.1743,  1.0756, -0.6372,  0.0164,  0.0300,  0.1283,  0.1555,
         -0.3714,  0.1329, -0.3921, -0.0087]], device='cuda:0')
Scaled actions :  tensor([[-0.2560,  0.1743,  1.0756, -0.6372,  0.0164,  0.0300,  0.1283,  0.1555,
         -0.3714,  0.1329, -0.3921, -0.0087]], device='cuda:0')
obs :  tensor([[ 4.1064e-02, -1.5364e-01, -4.3014e-01, -4.6797e-02, -3.7363e-03,
         -9.9890e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.8992e-02,
         -7.8086e-03,  1.4638e-01, -4.5007e-02, -2.2553e-01, -3.0211e-02,
         -5.9308e-03,  4.4251e-02,  9.4072e-02, -1.8080e-02, -1.6129e-01,
         -8.3154e-02, -4.3441e-02, -2.5778e-02,  1.3565e-01, -8.4648e-02,
          1.7142e-03,  2.7264e-02,  2.3638e-02,  4.5922e-02,  4.8463e-04,
          3.4415e-02,  8.0124e-02, -7.0228e-03, -2.5605e-01,  1.7426e-01,
          1.0756e+00, -6.3721e-01,  1.6448e-02,  3.0011e-02,  1.2829e-01,
          1.5548e-01, -3.7143e-01,  1.3290e-01, -3.9212e-01, -8.7

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.3114,  0.0553,  1.1365, -0.5741, -0.2217,  0.0626,  0.1151,  0.2552,
         -0.3447, -0.0150, -0.3836,  0.1985]], device='cuda:0')
Scaled actions :  tensor([[-0.3114,  0.0553,  1.1365, -0.5741, -0.2217,  0.0626,  0.1151,  0.2552,
         -0.3447, -0.0150, -0.3836,  0.1985]], device='cuda:0')
obs :  tensor([[ 0.0165, -0.1307, -0.5128, -0.0526, -0.0058, -0.9986,  1.0000,  0.0000,
          0.0000, -0.0382, -0.0118,  0.1743, -0.0637, -0.2131, -0.0217,  0.0018,
          0.0529,  0.0884, -0.0072, -0.1426, -0.0759, -0.0449, -0.0138,  0.1380,
         -0.0983,  0.1118,  0.0493,  0.0465,  0.0398, -0.0534,  0.0659,  0.0957,
          0.0645, -0.3114,  0.0553,  1.1365, -0.5741, -0.2217,  0.0626,  0.1151,
          0.2552, -0.3447, -0.0150, -0.3836,  0.1985]], device='cuda:0')
torques: [  -8.07213048  125.34691771   56.00952355  -97.83941979  200.
   12.19691692   17.27579314  -67.02781055 -200.           19.71299834
   -2.84750681   90.67628208]
データ収集:

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.3609,  0.0182,  1.1218, -0.5169, -0.4463,  0.0664,  0.1004,  0.3118,
         -0.3511, -0.0633, -0.3595,  0.2179]], device='cuda:0')
Scaled actions :  tensor([[-0.3609,  0.0182,  1.1218, -0.5169, -0.4463,  0.0664,  0.1004,  0.3118,
         -0.3511, -0.0633, -0.3595,  0.2179]], device='cuda:0')
obs :  tensor([[-0.0048, -0.0728, -0.5827, -0.0565, -0.0072, -0.9984,  1.0000,  0.0000,
          0.0000, -0.0481, -0.0142,  0.2016, -0.0844, -0.1814, -0.0105,  0.0117,
          0.0609,  0.0725,  0.0049, -0.1241, -0.0539, -0.0519, -0.0092,  0.1328,
         -0.1047,  0.1946,  0.0578,  0.0500,  0.0395, -0.1016,  0.0556,  0.0876,
          0.1372, -0.3609,  0.0182,  1.1218, -0.5169, -0.4463,  0.0664,  0.1004,
          0.3118, -0.3511, -0.0633, -0.3595,  0.2179]], device='cuda:0')
torques: [ -11.46010585   65.02033764   38.54390983  -18.1118439    88.75397054
  -64.08607342  -13.93892913  -32.20920415 -200.          -71.94983524
  -29.23700174   79.8410774

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[-0.3961,  0.0459,  1.2856, -0.4503, -0.5935,  0.0448,  0.0668,  0.3431,
         -0.2617, -0.0181, -0.2679,  0.1846]], device='cuda:0')
Scaled actions :  tensor([[-0.3961,  0.0459,  1.2856, -0.4503, -0.5935,  0.0448,  0.0668,  0.3431,
         -0.2617, -0.0181, -0.2679,  0.1846]], device='cuda:0')
obs :  tensor([[-0.0320, -0.0153, -0.6247, -0.0580, -0.0077, -0.9983,  1.0000,  0.0000,
          0.0000, -0.0604, -0.0154,  0.2278, -0.1054, -0.1414,  0.0019,  0.0216,
          0.0696,  0.0479,  0.0139, -0.1068, -0.0216, -0.0637, -0.0038,  0.1233,
         -0.0976,  0.1932,  0.0594,  0.0458,  0.0456, -0.1366,  0.0333,  0.0792,
          0.1694, -0.3961,  0.0459,  1.2856, -0.4503, -0.5935,  0.0448,  0.0668,
          0.3431, -0.2617, -0.0181, -0.2679,  0.1846]], device='cuda:0')
torques: [  69.35498779   49.11596161 -148.57623753  156.88987825 -200.
  -95.04126856  -90.49300043  -73.86161595   -8.03777111 -136.09085423
 -133.52608448 -191.36005408]
データ収集

In [40]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.508, Scaled action max=1.508
Step 1/10, Total steps: 182
steps: 182
actions : tensor([[ 0.8881,  1.5080, -0.8545, -1.4155, -1.0855,  0.4336,  0.8933, -0.1665,
          1.3923,  0.5789,  1.0911, -1.1725]], device='cuda:0')
target_dof_pos: tensor([[ 0.2431,  0.3385, -1.0319,  1.2770, -0.9855,  0.1001,  0.2604, -0.0404,
         -0.4572,  1.6956, -0.6071, -0.2711]], device='cuda:0')
Step 1: Original action max=1.720, Scaled action max=1.720
Step 2: Original action max=1.887, Scaled action max=1.887
データ収集完了: 10 steps collected with action_scale=1.0


In [41]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [42]:
env.sim.stop()

In [36]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_fixed.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_fixed.csv
データ形状: (92, 58)
